# 02 — Aggregate to county and verify against EAGLE-I

Turn raw snapshots into EAGLE-I-shaped county-hour labels, map coverage, and
cross-check against EAGLE-I. Requires snapshots in `data/snapshots/` (run the
collector first) and the Census county shapefile in `data/counties/`.

## Setup

In [ ]:
import sys
from pathlib import Path

# Make the repo-root modules importable from this notebook (it lives in notebooks/).
root = Path.cwd()
while not (root / "config.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
print("repo root:", root)

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import config
from county_aggregate import load_snapshots, place_in_counties, aggregate_to_county_hour

## Step 1 — Load snapshots and aggregate to county-hour

Output columns match the EAGLE-I outage table: `fips, time_block,
customers_out, outage_flag`.

In [ ]:
raw = load_snapshots()                 # or load_snapshots(start="2026-09-01")
print(f"{len(raw):,} raw records")

placed = place_in_counties(raw)        # needs data/counties/ shapefile
county = aggregate_to_county_hour(placed)

# Save where county_aggregate.py writes it, so crosscheck.py can read it.
out_path = config.AGGREGATED_DIR / "scraped_county_outages.parquet"
county.to_parquet(out_path, index=False)
print(f"{len(county):,} county-hour rows -> {out_path}")
print(f"positive-label rate: {county['outage_flag'].mean():.3f}  "
      f"(EAGLE-I climatology ~0.136)")
county.head()

## Step 2 — Map coverage

Which counties does the registry actually observe? Grey counties are coverage
gaps -- utilities to add, or counties to exclude from verification.

In [ ]:
counties_gdf = gpd.read_file(config.COUNTY_SHAPEFILE)[["GEOID", "geometry"]]
counties_gdf = counties_gdf.rename(columns={"GEOID": "fips"})

peak = county.groupby("fips")["customers_out"].max().rename("peak_customers_out")
covered = counties_gdf.merge(peak, on="fips", how="left")

fig, ax = plt.subplots(figsize=(14, 8))
covered.plot(column="peak_customers_out", ax=ax, cmap="OrRd", legend=True,
             missing_kwds={"color": "lightgrey"},
             edgecolor="white", linewidth=0.1)
ax.set_title("Counties observed by the outage collector "
             "(grey = no data, a coverage gap)")
ax.set_axis_off()
plt.show()

## Step 3 — Cross-check against EAGLE-I

On any window where the scrape overlaps an EAGLE-I release, quantify the
coverage gap. Export EAGLE-I to a parquet with columns `fips, time_block,
customers_out` at hourly resolution first.

In [ ]:
from crosscheck import run

eaglei_path = "eaglei_county_hour.parquet"   # <-- change me

try:
    run(str(config.AGGREGATED_DIR / "scraped_county_outages.parquet"), eaglei_path)
except SystemExit as exc:
    print(exc)

---

Report the **coverage ratio** and **flag agreement** from Step 3 in the paper,
and consider restricting verification to counties where the scrape tracks
EAGLE-I closely. Remember the retrospective verification (archived HRRR 48 h
forecasts vs EAGLE-I 2025) is the guaranteed result -- this live path is the
operational layer on top of it.